# 01 · Data Cleaning
**Netflix Customer Intelligence, Churn Prediction & AI Analytics Copilot**

**Input:** 9 raw CSVs in `data/raw/` **Output:** 9 cleaned CSVs + `data_quality_log.csv` in `data/processed/`

**Principles**
1. Never overwrite silently: every fix is counted in a **data-quality log** at the bottom.
2. Raw values that we change are kept in `*_original` / `*_flag` audit columns inside the processed CSVs (the MySQL tables load only the schema columns).
3. No leakage: cleaning uses only the same customer's own history or global statistics, never the churn label.
4. "Today" for this synthetic dataset is **2026-06-30** (latest date in churn, viewing, tickets and feedback). All "last N months" logic downstream is anchored to it.

In [1]:
import pandas as pd, numpy as np
from pathlib import Path

RAW  = Path('../data/raw')
PROC = Path('../data/processed'); PROC.mkdir(parents=True, exist_ok=True)
REFERENCE_DATE = pd.Timestamp('2026-06-30')

dq_log = []
def log(table, issue, rows, action):
    dq_log.append({'table': table, 'issue': issue, 'rows_affected': int(rows), 'action': action})
    print(f'[{table}] {issue}: {int(rows):,} rows -> {action}')

def rd(name, **kw):
    return pd.read_csv(RAW / f'{name}.csv', **kw)

customers = rd('customers'); plans = rd('subscription_plans'); subs = rd('subscriptions')
content = rd('content'); viewing = rd('viewing_activity'); payments = rd('payments')
tickets = rd('support_tickets'); feedback = rd('customer_feedback'); churn = rd('churn_labels')

raw_shapes = {n: d.shape for n, d in dict(customers=customers, plans=plans, subs=subs, content=content,
              viewing=viewing, payments=payments, tickets=tickets, feedback=feedback, churn=churn).items()}
pd.DataFrame(raw_shapes, index=['rows','cols']).T

,rows,cols
customers,8048,12
plans,3,6
subs,8875,10
content,500,9
viewing,66422,9
payments,110735,8
tickets,6120,11
feedback,5046,6
churn,8000,4


## 0 · Generic hygiene: strip whitespace on all text columns

In [2]:
for name, df in dict(customers=customers, plans=plans, subs=subs, content=content, viewing=viewing,
                     payments=payments, tickets=tickets, feedback=feedback, churn=churn).items():
    for c in df.select_dtypes(include=['object','string']).columns:
        df[c] = df[c].str.strip()
print('done')

done


## 1 · `customers`
Issues from the dictionary: ~0.6% duplicates, inconsistent country spellings, missing age / country / city / language, impossible ages, `state_or_region` blank.

In [3]:
# 1a. Exact duplicate rows
before = len(customers)
customers = customers.drop_duplicates().reset_index(drop=True)
assert customers.customer_id.is_unique
log('customers', 'duplicate customer rows', before - len(customers), 'dropped exact duplicates (kept first)')

[customers] duplicate customer rows: 48 rows -> dropped exact duplicates (kept first)


In [4]:
# 1b. Canonical country names. Built from an alias table; anything unmapped raises an error.
CANON = {
 'United States':['united states','us','usa','u.s.a','u.s.'],
 'India':['india','in','bharat'],
 'United Kingdom':['united kingdom','uk','u.k.','britain'],
 'Canada':['canada','ca'], 'Brazil':['brazil','br'], 'Germany':['germany','de'],
 'France':['france','fr'], 'Mexico':['mexico','mx'], 'Japan':['japan','jp'],
 'Philippines':['philippines','ph'], 'Indonesia':['indonesia','id'], 'Italy':['italy','it'],
 'Nigeria':['nigeria','ng'], 'Spain':['spain','es'], 'South Africa':['south africa','za'],
 'Australia':['australia','au'], 'South Korea':['south korea','kr','s. korea'],
 'Argentina':['argentina','ar'], 'Poland':['poland','pl'],
 'United Arab Emirates':['united arab emirates','uae','u.a.e','ae'],
}
COUNTRY_MAP = {a: canon for canon, aliases in CANON.items() for a in aliases}

def clean_country(s):
    out = s.str.lower().map(COUNTRY_MAP)
    unmapped = s[s.notna() & out.isna()].unique()
    assert len(unmapped) == 0, f'Unmapped country values: {unmapped}'
    return out

n_before = customers.country.nunique()
changed = (customers.country.notna() & (customers.country != clean_country(customers.country))).sum()
customers['country'] = clean_country(customers.country)
log('customers', f'inconsistent country spellings ({n_before} variants)', changed, f'standardised to {customers.country.nunique()} canonical countries')

[customers] inconsistent country spellings (83 variants): 1,035 rows -> standardised to 20 canonical countries


In [5]:
# 1c. Missing country: infer from the customer's own most frequent login_location (viewing_activity),
#     otherwise 'Unknown'. (Uses behaviour, never the churn label.)
viewing['login_location'] = clean_country(viewing.login_location)
loc_mode = (viewing.dropna(subset=['login_location']).groupby('customer_id').login_location
            .agg(lambda s: s.value_counts().idxmax()))
miss = customers.country.isna()
customers.loc[miss, 'country'] = customers.loc[miss, 'customer_id'].map(loc_mode)
still = customers.country.isna()
customers.loc[still, 'country'] = 'Unknown'
log('customers', 'missing country', miss.sum(), f'{miss.sum()-still.sum()} inferred from most-frequent login_location, {still.sum()} set to "Unknown"')

[customers] missing country: 409 rows -> 409 inferred from most-frequent login_location, 0 set to "Unknown"


In [6]:
# 1d. Age: valid range 13-100 (13 is the platform's minimum). Out-of-range -> NaN. Then impute median by customer_segment.
invalid = (customers.age < 13) | (customers.age > 100)
log('customers', 'implausible age (<13 or >100)', invalid.sum(), 'set to missing before imputation')
customers['age_original'] = customers.age
customers.loc[invalid, 'age'] = np.nan
n_missing = customers.age.isna().sum()
customers['age_was_imputed'] = customers.age.isna()
customers['age'] = customers.age.fillna(customers.groupby('customer_segment').age.transform('median'))
customers['age'] = customers.age.round().astype('Int64')
log('customers', 'missing age (incl. invalid)', n_missing, 'imputed with median age of the customer_segment')

[customers] implausible age (<13 or >100): 21 rows -> set to missing before imputation
[customers] missing age (incl. invalid): 342 rows -> imputed with median age of the customer_segment


In [7]:
# 1e. City & language
m = customers.city.isna().sum()
customers['city'] = customers.city.fillna('Unknown')
log('customers', 'missing city', m, 'set to "Unknown" (no reliable imputation; city is not used in modelling)')

lang_mode = customers[customers.preferred_language.notna()].groupby('country').preferred_language.agg(lambda s: s.value_counts().idxmax())
m = customers.preferred_language.isna()
customers.loc[m, 'preferred_language'] = customers.loc[m, 'country'].map(lang_mode)
left = customers.preferred_language.isna()
customers.loc[left, 'preferred_language'] = 'Unknown'
log('customers', 'missing preferred_language', m.sum(), f'imputed with most common language of the customer\'s country ({left.sum()} -> "Unknown")')

log('customers', 'state_or_region blank for all rows', len(customers), 'left NULL (documented as reserved/empty column)')
customers['registration_date'] = pd.to_datetime(customers.registration_date)
print(customers.isna().sum()[lambda s: s > 0])
customers.country.value_counts().head(25)

[customers] missing city: 483 rows -> set to "Unknown" (no reliable imputation; city is not used in modelling)
[customers] missing preferred_language: 251 rows -> imputed with most common language of the customer's country (0 -> "Unknown")
[customers] state_or_region blank for all rows: 8,000 rows -> left NULL (documented as reserved/empty column)
state_or_region    8000
age_original        321
dtype: int64


country
United States           1520
India                   1322
United Kingdom           649
Canada                   433
Germany                  401
Brazil                   397
France                   329
Mexico                   313
Japan                    291
Indonesia                249
Philippines              247
Italy                    231
Nigeria                  228
South Africa             228
Spain                    219
South Korea              215
Australia                212
Argentina                174
Poland                   171
United Arab Emirates     171
Name: count, dtype: int64

## 2 · `subscription_plans`, `subscriptions`, `content`

In [8]:
for c in ['subscription_start_date','subscription_end_date','cancellation_date']:
    subs[c] = pd.to_datetime(subs[c], errors='coerce')

assert subs.subscription_id.is_unique
assert subs.plan_id.isin(plans.plan_id).all()
bad_order = (subs.subscription_end_date < subs.subscription_start_date).sum()
log('subscriptions', 'end date before start date', bad_order, 'none found' if bad_order == 0 else 'review')
before_reg = subs.merge(customers[['customer_id','registration_date']], on='customer_id')
n = (before_reg.subscription_start_date < before_reg.registration_date).sum()
log('subscriptions', 'subscription starts before registration_date', n, 'reported only (not modified)')
# Blank end_date = still ongoing (NOT missing) so we leave it NULL.
log('subscriptions', 'blank subscription_end_date (ongoing)', subs.subscription_end_date.isna().sum(), 'kept NULL: means still ongoing')
print(subs.subscription_status.value_counts().to_dict())
print(plans)

[subscriptions] end date before start date: 0 rows -> none found
[subscriptions] subscription starts before registration_date: 0 rows -> reported only (not modified)
[subscriptions] blank subscription_end_date (ongoing): 8,000 rows -> kept NULL: means still ongoing
{'Active': 6009, 'Cancelled': 1991, 'Plan Changed': 875}
  plan_id plan_name  monthly_price    video_quality  max_devices  \
0  PLAN01     Basic           6.99        720p (HD)            1   
1  PLAN02  Standard          15.49  1080p (Full HD)            2   
2  PLAN03   Premium          22.99    4K (Ultra HD)            4   

  advertisements  
0            Yes  
1             No  
2             No  


In [9]:
assert content.content_id.is_unique and content.isna().sum().sum() == 0
print(content.content_type.value_counts().to_dict())

{'Movie': 216, 'TV Show': 144, 'Limited Series': 59, 'Documentary': 55, 'Stand-Up Special': 26}


## 3 · `viewing_activity`
Issues: inconsistent device labels (~20%), missing device & completion %, extreme watch-time outliers, login_location spellings.

In [10]:
# 3a. Device canonicalisation
DEVICE_MAP = {
 'smart tv':'Smart TV','smarttv':'Smart TV','tv':'Smart TV',
 'mobile':'Mobile','smartphone':'Mobile','phone':'Mobile',
 'tablet':'Tablet','ipad':'Tablet',
 'laptop':'Laptop','desktop':'Laptop','pc':'Laptop',
 'gaming console':'Gaming Console','console':'Gaming Console','playstation/xbox':'Gaming Console',
}
raw_dev = viewing.device_type
mapped = raw_dev.str.lower().map(DEVICE_MAP)
assert not (raw_dev.notna() & mapped.isna()).any(), raw_dev[raw_dev.notna() & mapped.isna()].unique()
log('viewing_activity', f'inconsistent device labels ({raw_dev.nunique()} variants)', (raw_dev.notna() & (raw_dev != mapped)).sum(), f'mapped to {mapped.nunique()} canonical devices')
viewing['device_type'] = mapped

[viewing_activity] inconsistent device labels (23 variants): 10,285 rows -> mapped to 5 canonical devices


In [11]:
# 3b. Missing device: fill with that customer's most-used device, otherwise "Unknown"
dev_mode = viewing.dropna(subset=['device_type']).groupby('customer_id').device_type.agg(lambda s: s.value_counts().idxmax())
m = viewing.device_type.isna()
viewing.loc[m, 'device_type'] = viewing.loc[m, 'customer_id'].map(dev_mode)
left = viewing.device_type.isna(); viewing.loc[left, 'device_type'] = 'Unknown'
log('viewing_activity', 'missing device_type', m.sum(), f"filled with customer's most-used device ({left.sum()} -> 'Unknown')")

[viewing_activity] missing device_type: 664 rows -> filled with customer's most-used device (1 -> 'Unknown')


In [12]:
# 3c. Missing completion_percentage: median by content_type; clip to 0-100
viewing = viewing.merge(content[['content_id','content_type']], on='content_id', how='left')
m = viewing.completion_percentage.isna()
viewing['completion_percentage'] = viewing.completion_percentage.fillna(viewing.groupby('content_type').completion_percentage.transform('median'))
viewing['completion_percentage'] = viewing.completion_percentage.clip(0, 100).round(2)
viewing = viewing.drop(columns='content_type')
log('viewing_activity', 'missing completion_percentage', m.sum(), 'imputed with median of the content_type')

[viewing_activity] missing completion_percentage: 531 rows -> imputed with median of the content_type


In [13]:
# 3d. Outliers & logical consistency
viewing['viewing_date'] = pd.to_datetime(viewing.viewing_date)
viewing['watch_duration_original'] = viewing.watch_duration_minutes
viewing['session_duration_original'] = viewing.session_duration_minutes
print(viewing.watch_duration_minutes.quantile([.5,.9,.99,.999,1]).round(0).to_dict())

WATCH_CAP, SESSION_CAP = 240, 300       # 4h single title, 5h session; the data shows a clear gap at ~108 -> 500+ min
out_w = viewing.watch_duration_minutes > WATCH_CAP
viewing['is_watch_outlier'] = out_w
viewing.loc[out_w, 'watch_duration_minutes'] = WATCH_CAP
log('viewing_activity', f'extreme watch_duration (>{WATCH_CAP} min, 500-900 binge sessions)', out_w.sum(), f'capped at {WATCH_CAP} (winsorised, flag kept in is_watch_outlier)')

out_s = viewing.session_duration_minutes > SESSION_CAP
viewing.loc[out_s, 'session_duration_minutes'] = SESSION_CAP
log('viewing_activity', f'extreme session_duration (>{SESSION_CAP} min)', out_s.sum(), f'capped at {SESSION_CAP}')

inc = viewing.watch_duration_minutes > viewing.session_duration_minutes
viewing.loc[inc, 'session_duration_minutes'] = viewing.loc[inc, 'watch_duration_minutes']
log('viewing_activity', 'watch_duration > session_duration (impossible, session includes pauses)', inc.sum(), 'session_duration raised to watch_duration')

viewing['watch_duration_minutes'] = viewing.watch_duration_minutes.astype(int)
viewing['session_duration_minutes'] = viewing.session_duration_minutes.astype(int)
assert viewing.viewing_id.is_unique

{0.5: 48.0, 0.9: 80.0, 0.99: 108.0, 0.999: 736.0, 1.0: 894.0}
[viewing_activity] extreme watch_duration (>240 min, 500-900 binge sessions): 161 rows -> capped at 240 (winsorised, flag kept in is_watch_outlier)
[viewing_activity] extreme session_duration (>300 min): 161 rows -> capped at 300
[viewing_activity] watch_duration > session_duration (impossible, session includes pauses): 6,068 rows -> session_duration raised to watch_duration


## 4 · `payments`
Required: reconcile the blank `subscription_id`, and handle anomalous amounts.

In [14]:
payments['payment_date'] = pd.to_datetime(payments.payment_date)
assert payments.payment_id.is_unique

# 4a. Resolve subscription_id: customer_id + payment_date inside [start, end] (blank end = ongoing)
s = subs[['subscription_id','customer_id','subscription_start_date','subscription_end_date','monthly_price']].copy()
s['end_filled'] = s.subscription_end_date.fillna(pd.Timestamp('2100-01-01'))
m = payments[['payment_id','customer_id','payment_date']].merge(s, on='customer_id')
m = m[(m.payment_date >= m.subscription_start_date) & (m.payment_date <= m.end_filled)]
dups = m.payment_id.duplicated(keep=False).sum()
# On a plan-change boundary day two windows overlap -> take the subscription that started latest (the new plan)
m = m.sort_values(['payment_id','subscription_start_date']).drop_duplicates('payment_id', keep='last')
payments = payments.drop(columns='subscription_id').merge(m[['payment_id','subscription_id','monthly_price']].rename(columns={'monthly_price':'matched_sub_price'}), on='payment_id', how='left')
log('payments', 'subscription_id blank on every row', len(payments), f'resolved {payments.subscription_id.notna().sum():,} via customer+date window ({dups} rows hit overlapping boundary windows -> picked the newer subscription)')
assert payments.subscription_id.notna().all(), 'unmatched payments found'

[payments] subscription_id blank on every row: 110,735 rows -> resolved 110,735 via customer+date window (18 rows hit overlapping boundary windows -> picked the newer subscription)


In [15]:
# 4b. Anomalous amounts.
# Legitimate amounts for a customer = any of THEIR plan prices, half of a plan price (proration), or 0 (waiver/refund).
# Everything else is an error / outlier and is restated to the matched subscription's price.
prices = subs.sort_values('subscription_start_date').groupby('customer_id').monthly_price.apply(list)
p1 = payments.customer_id.map(prices.str[0]); p2 = payments.customer_id.map(prices.str[-1])
def near(a, b): return (a - b).abs() <= 0.011
valid = (payments.amount == 0) | near(payments.amount, p1) | near(payments.amount, p2) | near(payments.amount, p1/2) | near(payments.amount, p2/2)
payments['amount_original'] = payments.amount
payments['amount_flag'] = np.where(payments.amount == 0, 'zero_amount',
                           np.where(valid & ~(near(payments.amount,p1)|near(payments.amount,p2)), 'proration_half_price',
                           np.where(~valid, 'anomalous_restated', 'ok')))
payments.loc[~valid, 'amount'] = payments.loc[~valid, 'matched_sub_price']
print(payments.amount_flag.value_counts())
log('payments', 'anomalous amounts (e.g. 2x/3x price, 76.89, 170.39)', (~valid).sum(), 'restated to matched subscription price; original in amount_original')
log('payments', 'zero-amount payments', (payments.amount_flag=='zero_amount').sum(), 'kept as-is (waiver/refund), flagged')
log('payments', 'half-price payments', (payments.amount_flag=='proration_half_price').sum(), 'kept as-is (proration), flagged')
payments = payments.drop(columns='matched_sub_price')

amount_flag
ok                      109623
anomalous_restated         656
zero_amount                250
proration_half_price       206
Name: count, dtype: int64
[payments] anomalous amounts (e.g. 2x/3x price, 76.89, 170.39): 656 rows -> restated to matched subscription price; original in amount_original
[payments] zero-amount payments: 250 rows -> kept as-is (waiver/refund), flagged
[payments] half-price payments: 206 rows -> kept as-is (proration), flagged


In [16]:
# 4c. Consistency of status vs failure reason
bad = ((payments.payment_status=='Failed') != payments.failed_payment_reason.notna()).sum()
log('payments', 'status/failed_reason inconsistent', bad, 'none found' if bad == 0 else 'review')
print(payments.payment_status.value_counts().to_dict()); print(payments.payment_date.min(), payments.payment_date.max())

[payments] status/failed_reason inconsistent: 0 rows -> none found
{'Success': 103529, 'Failed': 7206}
2019-01-01 00:00:00 2026-05-31 00:00:00


## 5 · `support_tickets`

In [17]:
raw_dates = tickets.ticket_date.copy()
tickets['ticket_date'] = pd.to_datetime(tickets.ticket_date, errors='coerce')
bad = tickets.ticket_date.isna()
print(raw_dates[bad].value_counts().to_dict())
tickets['ticket_date_invalid'] = bad
log('support_tickets', 'malformed ticket_date (e.g. 2025-13-45)', bad.sum(), 'set to NULL, flagged in ticket_date_invalid (cannot be recovered safely; excluded from time-based queries)')

# CSAT / resolution time blank only for open tickets (structurally missing, NOT imputed)
closed = tickets.ticket_status.isin(['Resolved','Closed'])
assert tickets.loc[closed, 'customer_satisfaction_score'].notna().all() and tickets.loc[closed,'resolution_time_hours'].notna().all()
assert tickets.loc[~closed, 'customer_satisfaction_score'].isna().all()
log('support_tickets', 'blank CSAT/resolution_time on Pending/Escalated', (~closed).sum(), 'expected, kept NULL (never imputed: filling would invent satisfaction scores)')
tickets['customer_satisfaction_score'] = tickets.customer_satisfaction_score.astype('Int64')
print(tickets.resolution_time_hours.describe().round(1).to_dict())

{'2025-13-45': 61}
[support_tickets] malformed ticket_date (e.g. 2025-13-45): 61 rows -> set to NULL, flagged in ticket_date_invalid (cannot be recovered safely; excluded from time-based queries)
[support_tickets] blank CSAT/resolution_time on Pending/Escalated: 1,528 rows -> expected, kept NULL (never imputed: filling would invent satisfaction scores)
{'count': 4592.0, 'mean': 18.0, 'std': 17.7, 'min': 0.5, '25%': 5.2, '50%': 12.7, '75%': 25.0, 'max': 151.1}


## 6 · `customer_feedback`
The dictionary warns ~7% of `sentiment_label` values are wrong. Validation: the feedback text has only 13 distinct templates, so I check the label against **(a)** the rating and **(b)** the majority label of identical texts.

In [18]:
feedback['feedback_date'] = pd.to_datetime(feedback.feedback_date)
by_rating = feedback.rating.map(lambda r: 'Negative' if r <= 2 else 'Neutral' if r == 3 else 'Positive')
text_major = feedback.groupby('feedback_text').sentiment_label.transform(lambda s: s.value_counts().idxmax())
print('rating-derived label == text-majority label:', (by_rating == text_major).mean())   # 1.0 -> two independent signals agree
feedback['sentiment_label_original'] = feedback.sentiment_label
wrong = feedback.sentiment_label != by_rating
feedback['sentiment_label'] = by_rating
log('customer_feedback', 'sentiment_label contradicts rating AND text', wrong.sum(), 'relabelled from rating (validated against text); original kept in sentiment_label_original')
print(f'{wrong.mean():.1%} relabelled')

rating-derived label == text-majority label: 1.0
[customer_feedback] sentiment_label contradicts rating AND text: 263 rows -> relabelled from rating (validated against text); original kept in sentiment_label_original
5.2% relabelled


## 7 · `churn_labels` and cross-table checks

In [19]:
churn['churn_date'] = pd.to_datetime(churn.churn_date)
assert churn.customer_id.is_unique and churn.customer_id.isin(customers.customer_id).all()
assert set(customers.customer_id) == set(churn.customer_id)
assert ((churn.churned) == churn.churn_date.notna()).all() and ((churn.churned) == churn.churn_reason.notna()).all()
cancelled = set(subs.loc[subs.subscription_status=='Cancelled','customer_id'])
agree = set(churn.loc[churn.churned,'customer_id']) == cancelled
print('churn rate:', round(churn.churned.mean(), 4), '| churned == customers with a Cancelled subscription:', agree)
print('latest churn_date:', churn.churn_date.max().date())
log('churn_labels', 'cross-check vs subscriptions.Cancelled', len(cancelled), 'churn labels fully consistent' if agree else 'MISMATCH, investigate')

churn rate: 0.2489 | churned == customers with a Cancelled subscription: True
latest churn_date: 2026-06-30
[churn_labels] cross-check vs subscriptions.Cancelled: 1,991 rows -> churn labels fully consistent


In [20]:
# Referential integrity: every child table must point at an existing customer / content / plan
fk = {
 'subscriptions.customer_id': subs.customer_id.isin(customers.customer_id),
 'viewing.customer_id': viewing.customer_id.isin(customers.customer_id),
 'viewing.content_id': viewing.content_id.isin(content.content_id),
 'payments.customer_id': payments.customer_id.isin(customers.customer_id),
 'tickets.customer_id': tickets.customer_id.isin(customers.customer_id),
 'feedback.customer_id': feedback.customer_id.isin(customers.customer_id),
}
for k, ok in fk.items():
    print(f'{k:28s} orphans: {(~ok).sum()}')
    assert ok.all()

subscriptions.customer_id    orphans: 0
viewing.customer_id          orphans: 0
viewing.content_id           orphans: 0
payments.customer_id         orphans: 0
tickets.customer_id          orphans: 0
feedback.customer_id         orphans: 0


## 8 · Save cleaned data + data-quality log

In [21]:
out = dict(customers=customers, subscription_plans=plans, subscriptions=subs, content=content,
           viewing_activity=viewing, payments=payments, support_tickets=tickets,
           customer_feedback=feedback, churn_labels=churn)
for name, df in out.items():
    df.to_csv(PROC / f'{name}.csv', index=False)
dq = pd.DataFrame(dq_log); dq.to_csv(PROC / 'data_quality_log.csv', index=False)

summary = pd.DataFrame({'raw_rows': [raw_shapes[k][0] for k in ['customers','plans','subs','content','viewing','payments','tickets','feedback','churn']],
                        'clean_rows': [len(d) for d in out.values()]}, index=out.keys())
display(summary)
dq

,raw_rows,clean_rows
customers,8048,8000
subscription_plans,3,3
subscriptions,8875,8875
content,500,500
viewing_activity,66422,66422
payments,110735,110735
support_tickets,6120,6120
customer_feedback,5046,5046
churn_labels,8000,8000


,table,issue,rows_affected,action
0,customers,duplicate customer rows,48,dropped exact duplicates (kept first)
1,customers,inconsistent country spellings (83 variants),1035,standardised to 20 canonical countries
2,customers,missing country,409,409 inferred from most-frequent login_location...
3,customers,implausible age (<13 or >100),21,set to missing before imputation
4,customers,missing age (incl. invalid),342,imputed with median age of the customer_segment
5,customers,missing city,483,"set to ""Unknown"" (no reliable imputation; city..."
6,customers,missing preferred_language,251,imputed with most common language of the custo...
7,customers,state_or_region blank for all rows,8000,left NULL (documented as reserved/empty column)
8,subscriptions,end date before start date,0,none found
9,subscriptions,subscription starts before registration_date,0,reported only (not modified)
